Synchronized On-Demand Advertising (SODA) <br>
Written by: Robert Methven (@methvenr) <br>
Date: April 2025 <br>

lay between searches
    time.
1. Install required libraries:
```bash
pip install opencv-python torch torchvision selenium
```

2. Make sure you have ChromeDriver installed for Selenium

3. Replace "path_to_your_video.mp4" with your actual video path

This version of the code:
- Processes a video file frame by frame
- Detects clothing items using a pre-trained model
- Searches for detected items on Google Shopping
- Prints the name, price, and link for found items

The code is more straightforward but less modular than the previous version. Keep in mind:
- It might be less maintainable for larger projects
- Error handling is minimal
- You might need to adjust the confidence threshold (0.7) based on your needs
- The frame sampling rate (every 30 frames) can be adjusted
- Add more delay between searches if needed to avoid being blocked
- The web element class names might need updates if Google changes their website structure

Remember to comply with the terms of service of any shopping websites you're searching.

In [ ]:
pip install opencv-python torch torchvision selenium

In [ ]:
import cv2
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
import pandas as pd
import datetime

In [ ]:
# Initialize the model
model = fasterrcnn_resnet50_fpn(pretrained=True)
model.eval()

In [ ]:
# Define clothing categories
categories = [
    'short sleeve top', 'long sleeve top', 'short sleeve outwear',
    'long sleeve outwear', 'vest', 'sling', 'shorts', 'trousers',
    'skirt', 'short sleeve dress', 'long sleeve dress', 'vest dress',
    'sling dress'
]

In [ ]:
# Initialize web driver
driver = webdriver.Chrome()

In [ ]:
# Initialize lists for dataframe
all_items = []
all_timestamps = []
all_descriptions = []
all_links = []
all_prices = []

In [ ]:
# Video processing
video_path = "SODA files/roc.mp4"
cap = cv2.VideoCapture(video_path)
frame_count = 0
fps = cap.get(cv2.CAP_PROP_FPS)
detected_items_dict = {}  # Dictionary to store items and their timestamps

In [ ]:
# Validate categories
if not categories or not isinstance(categories, (list, tuple)):
    raise ValueError("Categories must be a non-empty list or tuple")

detected_items_dict = {}
frame_count = 0
fps = cap.get(cv2.CAP_PROP_FPS)

try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        current_time = frame_count / fps
        timestamp = str(datetime.timedelta(seconds=int(current_time)))

        if frame_count % 30 == 0:
            # Move tensor creation to GPU if available
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            img_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
            img_tensor = img_tensor.unsqueeze(0).to(device)

            with torch.no_grad():
                predictions = model(img_tensor)

            # Process predictions
            scores = predictions[0]['scores'].cpu()  # Move to CPU if on GPU
            labels = predictions[0]['labels'].cpu()  # Move to CPU if on GPU
            
            # Use numpy for faster processing
            confident_mask = scores > 0.7  # Confidence threshold
            confident_labels = labels[confident_mask]
            
            for label in confident_labels:
                # Convert tensor to integer and validate index
                label_idx = int(label.item()) - 1
                if 0 <= label_idx < len(categories):
                    detected_item = categories[label_idx]
                    if detected_item not in detected_items_dict:
                        detected_items_dict[detected_item] = []
                    detected_items_dict[detected_item].append(timestamp)
                else:
                    print(f"Warning: Invalid label index {label_idx} for categories list of length {len(categories)}")

        frame_count += 1

except Exception as e:
    print(f"Error processing frame {frame_count}: {str(e)}")
    raise

finally:
    cap.release()


In [ ]:
# Add this at the start
detected_unknown_labels = set()

# Process video frames
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Calculate current timestamp
    current_time = frame_count / fps
    timestamp = str(datetime.timedelta(seconds=int(current_time)))

    # Process every 30 frames to save resources
    if frame_count % 30 == 0:
        # Convert frame to tensor
        img_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
        img_tensor = img_tensor.unsqueeze(0)

        # Get predictions
        with torch.no_grad():
            predictions = model(img_tensor)

        # Extract detected clothing items
        for score, label in zip(predictions[0]['scores'], predictions[0]['labels']):
            if score > 0.7:  # Confidence threshold
                label_idx = int(label.item()) - 1
                if 0 <= label_idx < len(categories):
                    detected_item = categories[label_idx]
                    if detected_item not in detected_items_dict:
                        detected_items_dict[detected_item] = []
                    detected_items_dict[detected_item].append(timestamp)
                else:
                    detected_unknown_labels.add(label_idx)

    frame_count += 1

cap.release()

# After processing is complete, print unknown labels
print(f"Detected labels outside category range: {sorted(detected_unknown_labels)}")


In [ ]:
# Search for each detected item and build dataframe
for item, timestamps in detected_items_dict.items():
    print(f"\nSearching for: {item}")
    
    # Search on Google Shopping
    driver.get("https://www.google.com/shopping")
    search_box = driver.find_element(By.NAME, "q")
    search_box.send_keys(item)
    search_box.send_keys(Keys.RETURN)
    
    # Wait for results to load
    time.sleep(2)
    
    # Get results
    items = driver.find_elements(By.CLASS_NAME, "sh-dgr__grid-result")
    
    # Process first 5 results for this item
    for i, search_result in enumerate(items[:5]):
        try:
            name = search_result.find_element(By.CLASS_NAME, "tAxDx").text
            price = search_result.find_element(By.CLASS_NAME, "a8Pemb").text
            link = search_result.find_element(By.TAG_NAME, "a").get_attribute("href")
            
            # Add to lists for dataframe
            all_items.append(item)
            all_timestamps.append(', '.join(timestamps))  # Join all timestamps for this item
            all_descriptions.append(name)
            all_links.append(link)
            all_prices.append(price)
            
        except:
            continue
    
    # Add delay between searches
    time.sleep(1)

In [ ]:
# Clean up
driver.quit()

In [ ]:
# Create DataFrame
df = pd.DataFrame({
    'Detected_Item': all_items,
    'Time_Detected': all_timestamps,
    'Product_Description': all_descriptions,
    'Price': all_prices,
    'Product_Link': all_links
})

In [ ]:
# Clean and format DataFrame
df = df.drop_duplicates()  # Remove any duplicate entries
df = df.reset_index(drop=True)

In [ ]:
# Save DataFrame to CSV
df.to_csv('SODA files/detected_clothing_items.csv', index=False)

In [ ]:
# Display summary
print("\nDetection Summary:")
print(df)

In [ ]:
# Print some statistics
print("\nStatistics:")
print(f"Total unique items detected: {len(detected_items_dict)}")
print(f"Total product matches found: {len(df)}")
print("\nDetections per item:")
for item, timestamps in detected_items_dict.items():
    print(f"{item}: {len(timestamps)} times")

In [ ]:
# Create a more detailed timestamp analysis
timestamp_df = pd.DataFrame({
    'Item': [item for item, ts_list in detected_items_dict.items() for _ in ts_list],
    'Exact_Timestamp': [ts for ts_list in detected_items_dict.values() for ts in ts_list]
})

In [ ]:
print("\nDetailed Timestamp Analysis:")
print(timestamp_df.sort_values('Exact_Timestamp'))

In [ ]:
# Optional: Create visualizations
try:
    import matplotlib.pyplot as plt
    
    # Plot number of detections per item
    detection_counts = timestamp_df['Item'].value_counts()
    plt.figure(figsize=(12, 6))
    detection_counts.plot(kind='bar')
    plt.title('Number of Detections per Item')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('detections_per_item.png')

except ImportError:
    print("Matplotlib not installed. Skipping visualizations.")

In [ ]:
pip install torch torchvision opencv-python selenium pandas

In [5]:
import cv2
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd
import datetime
import os
import torch

print("Loading YOLOv5 model...")

# Load YOLOv5 model
model = torch.hub.load('ultralytics/yolov5', 'yolov5s')

# Create directory for saving cropped images if it doesn't exist
output_dir = 'detected_items'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Video processing
video_path = "SODA files/roc.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video file")
    exit()

frame_count = 0
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Video FPS: {fps}")

detected_items = []

while True:
    ret, frame = cap.read()
    if not ret:
        print("End of video stream")
        break

    current_time = frame_count / fps
    timestamp = str(datetime.timedelta(seconds=int(current_time)))

    if frame_count % 30 == 0:  # Process every 30th frame
        print(f"Processing frame {frame_count}")
        
        try:
            # YOLOv5 inference
            results = model(frame)
            
            # Process detections
            for i, (x1, y1, x2, y2, conf, cls) in enumerate(results.xyxy[0]):
                class_name = model.names[int(cls)]
                if class_name == 'person' and conf > 0.5:
                    # Convert coordinates to integers
                    x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
                    
                    # Ensure coordinates are within frame boundaries
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)
                    
                    # Only process if we have a valid region
                    if x2 > x1 and y2 > y1:
                        # Crop the person
                        person_crop = frame[y1:y2, x1:x2].copy()
                        
                        # Create filename for this detection
                        image_filename = f'person_{frame_count}_{i}.jpg'
                        image_path = os.path.join(output_dir, image_filename)
                        
                        # Save the cropped image
                        cv2.imwrite(image_path, person_crop)
                        
                        # Store detection information
                        detected_items.append({
                            'timestamp': timestamp,
                            'confidence': float(conf),
                            'image_path': image_path
                        })
                        
                        print(f"Detected person at {timestamp} with confidence {conf:.2f}")
                        print(f"Saved image to: {image_path}")
                        
                        # Draw bounding box
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        cv2.putText(frame, 
                                  f"Person: {conf:.2f}", 
                                  (x1, y1 - 10), 
                                  cv2.FONT_HERSHEY_SIMPLEX, 
                                  0.5, 
                                  (0, 255, 0), 
                                  2)

            # Show the frame with detections
            cv2.imshow('Person Detections', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        except Exception as e:
            print(f"Error processing frame {frame_count}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue

    frame_count += 1

cap.release()
cv2.destroyAllWindows()

if not detected_items:
    print("No people detected in the video")
    exit()

print("\nVideo processing complete. Starting internet search...")

# Set up Chrome options for better stability
chrome_options = Options()
chrome_options.add_argument('--disable-notifications')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--start-maximized')

# Initialize lists for dataframe
all_timestamps = []
all_confidences = []
all_image_paths = []
all_similar_products = []
all_prices = []
all_links = []
all_sources = []  # To track whether product is from Amazon or eBay

def search_amazon(driver, search_term):
    results = []
    try:
        driver.get("https://www.amazon.com")
        
        # Find and use the search box
        search_box = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "twotabsearchtextbox"))
        )
        search_box.clear()
        search_box.send_keys(search_term)
        search_box.send_keys(Keys.RETURN)
        
        # Wait for results to load
        time.sleep(3)
        
        # Get product results
        products = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.s-result-item[data-component-type='s-search-result']"))
        )
        
        for product in products[:5]:  # Get first 5 products
            try:
                name = product.find_element(By.CSS_SELECTOR, "h2 span").text
                price_element = product.find_elements(By.CSS_SELECTOR, "span.a-price-whole")
                price = price_element[0].text if price_element else "Price not available"
                link = product.find_element(By.CSS_SELECTOR, "h2 a").get_attribute('href')
                
                results.append({
                    'name': name,
                    'price': f"${price}",
                    'link': link,
                    'source': 'Amazon'
                })
                
            except Exception as e:
                print(f"Error processing Amazon product: {str(e)}")
                continue
                
    except Exception as e:
        print(f"Error searching Amazon: {str(e)}")
    
    return results

def search_ebay(driver, search_term):
    results = []
    try:
        driver.get("https://www.ebay.com")
        
        # Find and use the search box
        search_box = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "gh-ac"))
        )
        search_box.clear()
        search_box.send_keys(search_term)
        search_box.send_keys(Keys.RETURN)
        
        # Wait for results to load
        time.sleep(3)
        
        # Get product results
        products = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li.s-item"))
        )
        
        for product in products[:5]:  # Get first 5 products
            try:
                name = product.find_element(By.CSS_SELECTOR, "h3.s-item__title").text
                price = product.find_element(By.CSS_SELECTOR, "span.s-item__price").text
                link = product.find_element(By.CSS_SELECTOR, "a.s-item__link").get_attribute('href')
                
                results.append({
                    'name': name,
                    'price': price,
                    'link': link,
                    'source': 'eBay'
                })
                
            except Exception as e:
                print(f"Error processing eBay product: {str(e)}")
                continue
                
    except Exception as e:
        print(f"Error searching eBay: {str(e)}")
    
    return results

# Process each detected item
for item in detected_items:
    print(f"\nSearching for items in image from timestamp {item['timestamp']}")
    
    # Initialize search terms based on detected person
    search_terms = [
        "clothing outfit similar to",
        "fashion clothes like",
        "similar style clothing to",
        "matching outfit to",
        "clothing style similar to"
    ]
    
    try:
        driver = webdriver.Chrome(options=chrome_options)
        
        # Search both Amazon and eBay for each search term
        for search_term in search_terms[:2]:  # Use first 2 search terms to avoid too many searches
            # Amazon search
            amazon_results = search_amazon(driver, search_term)
            for result in amazon_results:
                all_timestamps.append(item['timestamp'])
                all_confidences.append(item['confidence'])
                all_image_paths.append(item['image_path'])
                all_similar_products.append(result['name'])
                all_prices.append(result['price'])
                all_links.append(result['link'])
                all_sources.append(result['source'])
                
                print(f"\nFound on Amazon:")
                print(f"Name: {result['name']}")
                print(f"Price: {result['price']}")
                print(f"Link: {result['link']}")
            
            # eBay search
            ebay_results = search_ebay(driver, search_term)
            for result in ebay_results:
                all_timestamps.append(item['timestamp'])
                all_confidences.append(item['confidence'])
                all_image_paths.append(item['image_path'])
                all_similar_products.append(result['name'])
                all_prices.append(result['price'])
                all_links.append(result['link'])
                all_sources.append(result['source'])
                
                print(f"\nFound on eBay:")
                print(f"Name: {result['name']}")
                print(f"Price: {result['price']}")
                print(f"Link: {result['link']}")
                
    except Exception as e:
        print(f"Error with browser: {str(e)}")
        
    finally:
        try:
            driver.quit()
        except:
            pass
        
        time.sleep(2)

# Create DataFrame with results
if all_timestamps:
    df = pd.DataFrame({
        'Time_Detected': all_timestamps,
        'Confidence': all_confidences,
        'Image_Path': all_image_paths,
        'Similar_Product': all_similar_products,
        'Price': all_prices,
        'Product_Link': all_links,
        'Source': all_sources
    })

    # Save both full results and a summary
    df.to_csv('SODA files/detected_clothing_items_full.csv', index=False)
    
    # Create a summary with unique items
    summary_df = df.drop_duplicates(subset=['Similar_Product', 'Price', 'Source'])
    summary_df.to_csv('SODA files/detected_clothing_items_summary.csv', index=False)

    print("\nDetection Summary:")
    print(summary_df)

    print("\nFinal Statistics:")
    print(f"Total frames processed: {frame_count}")
    print(f"Total detections: {len(detected_items)}")
    print(f"Total unique products found: {len(summary_df)}")
    
    # Print all unique links grouped by source
    print("\nAll unique product links:")
    for source in ['Amazon', 'eBay']:
        source_links = df[df['Source'] == source]['Product_Link'].unique()
        print(f"\n{source} Links:")
        for i, link in enumerate(source_links, 1):
            print(f"{i}. {link}")
else:
    print("No products were found in the search results")


Loading YOLOv5 model...


Using cache found in C:\Users\methvenr/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2025-5-27 Python-3.11.5 torch-2.2.0+cu121 CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


Video FPS: 24.0
Processing frame 0
Detected person at 0:00:00 with confidence 0.75
Saved image to: detected_items\person_0_0.jpg
Processing frame 30
Processing frame 60
Processing frame 90
Detected person at 0:00:03 with confidence 0.52
Saved image to: detected_items\person_90_0.jpg
Processing frame 120
Detected person at 0:00:05 with confidence 0.55
Saved image to: detected_items\person_120_0.jpg
Processing frame 150
Processing frame 180
Processing frame 210
Detected person at 0:00:08 with confidence 0.84
Saved image to: detected_items\person_210_0.jpg
Processing frame 240
Processing frame 270
Processing frame 300
Detected person at 0:00:12 with confidence 0.88
Saved image to: detected_items\person_300_0.jpg
Processing frame 330
Processing frame 360
Processing frame 390
Detected person at 0:00:16 with confidence 0.60
Saved image to: detected_items\person_390_1.jpg
Processing frame 420
Detected person at 0:00:17 with confidence 0.65
Saved image to: detected_items\person_420_0.jpg
Proce

KeyboardInterrupt: 